In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.signal import savgol_filter
from scan_spectrolyser import scan_helpers
from scan_spectrolyser import no3_calibrations

In [ ]:
def fit_linear_function(linear_range, x, y):
    coeff = np.polyfit(linear_range[x], linear_range[y], 1)
    r2 = np.corrcoef(linear_range[x], linear_range[y])[0,1]**2
    return coeff, r2

def plot_linear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    coeff, r2 = fit_linear_function(linear_range, x, y)
    vals= np.polyval(coeff, data[x])
    ax.plot(data[x], vals, color='b')
    ax.text(text_start_x, text_start_y, 'Slope: %.4f' % (coeff[0]))
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.4f' % (coeff[1]))
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.4f' % (r2))
    return coeff, r2

# Chinese S::CAN

## New calibration with freshly made standards

In [ ]:
chinesecal_21425 = scan_helpers.import_scan_fp('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/SCAN Calibration 2-14-25/calibration.fp')
chinesecal_21425['Concentration (mg/L)'] = [0, .01, .05, .1, .5, 1, 2]
chinesecal_21425['Concentration (uM)'] = chinesecal_21425['Concentration (mg/L)']*1000/14
chinesecal_21425 = scan_helpers.apply_calibrations(chinesecal_21425, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})

chinesecal_21425_corrected = scan_helpers.correct_turbidity(chinesecal_21425.drop(['Concentration (mg/L)', 'Concentration (uM)', 'one_wavelength', 'two_wavelength', 'second_derivative'], axis=1))
chinesecal_21425_corrected['Concentration (mg/L)'] = [0, .01, .05, .1, .5, 1, 2]
chinesecal_21425_corrected['Concentration (uM)'] = chinesecal_21425['Concentration (mg/L)']*1000/14
chinesecal_21425_corrected = scan_helpers.apply_calibrations(chinesecal_21425_corrected, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})

In [ ]:
fig, ax = plt.subplots(nrows=3,ncols=2, figsize=(8.5,11))
chinesecal_21425.plot(y='Concentration (uM)', x='second_derivative', marker='.', linestyle = 'None', title = 'Uncorrected Second Derivative', ylabel = 'Concentration (uM)', legend=False, ax=ax[0,0])
plot_linear_fit(chinesecal_21425, chinesecal_21425[0:5], 'second_derivative', 'Concentration (uM)', .25, 60, 10, ax=ax[0,0])

chinesecal_21425.plot(y='Concentration (uM)', x='one_wavelength', marker='.', linestyle = 'None', title = 'Uncorrected One Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[1,0])
plot_linear_fit(chinesecal_21425, chinesecal_21425[0:5], 'one_wavelength', 'Concentration (uM)', .25, 60, 10, ax=ax[1,0])

chinesecal_21425.plot(y='Concentration (uM)', x='two_wavelength', marker='.', linestyle = 'None', title = 'Uncorrected Two Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[2,0])
plot_linear_fit(chinesecal_21425, chinesecal_21425[0:5], 'two_wavelength', 'Concentration (uM)', .25, 60, 10, ax=ax[2,0])

chinesecal_21425_corrected.plot(y='Concentration (uM)', x='second_derivative', marker='.', linestyle = 'None', title = 'Corrected Second Derivative', ylabel = 'Concentration (uM)', legend=False, ax=ax[0,1])
plot_linear_fit(chinesecal_21425_corrected, chinesecal_21425_corrected[0:5], 'second_derivative', 'Concentration (uM)', .25, 60, 10, ax=ax[0,1])

chinesecal_21425_corrected.plot(y='Concentration (uM)', x='one_wavelength', marker='.', linestyle = 'None', title = 'Corrected One Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[1,1])
plot_linear_fit(chinesecal_21425_corrected, chinesecal_21425_corrected[0:5], 'one_wavelength', 'Concentration (uM)', .25, 50, 10, ax=ax[1,1])

chinesecal_21425_corrected.plot(y='Concentration (uM)', x='two_wavelength', marker='.', linestyle = 'None', title = 'Corrected Two Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[2,1])
plot_linear_fit(chinesecal_21425_corrected, chinesecal_21425_corrected[0:5], 'two_wavelength', 'Concentration (uM)', .25, 60, 10, ax=ax[2,1])


fig.suptitle('Chinese S::CAN Calbrations')
fig.tight_layout()

- This calibration looks great! 
- linear up until 0.5 mg/L, non linear above that - will have to check with turbidity if still linear.
- for reference, in Reynolds we were seeing ~ 10 uM (.14 mg/L). So this range seems very reasonable
- correction has little effect on 2nd derivative calibration
- correction seriously messes up one-wavelength, but stil works really well for two_wavelength - even more linear than second derivative, up to 80 uM (1.1 uM)s

## Calibration made with old standards
- these are based off of erroneous SEAL standard

In [ ]:
chinesecal_21425_old = scan_helpers.import_scan_fp('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/SCAN Calibration 2-14-25/calibration_old.fp')
chinesecal_21425_old['Concentration (mg/L)'] = [0, .01, .05, .1, .5, 1, 2]
chinesecal_21425_old['Concentration (mg/L) Correct'] = chinesecal_21425_old['Concentration (mg/L)']*1.168
chinesecal_21425_old = scan_helpers.apply_calibrations(chinesecal_21425_old, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})

chinesecal_21425_old

In [ ]:
fig, ax = plt.subplots()
chinesecal_21425_old.plot(y='Concentration (mg/L) Correct', x='second_derivative', marker='.', linestyle = 'None', title = 'Chinese S::CAN Old Calibration', ylabel = 'Concentration (mg/L)', label='Old', ax=ax)
plot_linear_fit(chinesecal_21425_old, chinesecal_21425_old[0:5], 'second_derivative', 'Concentration (mg/L) Correct', .25, 1.5, .1, ax=ax)

chinesecal_21425.plot(y='Concentration (mg/L)', x='second_derivative', marker='.', linestyle = 'None', title = 'Chinese S::CAN New Calibration', ylabel = 'Concentration (mg/L)', label='New', ax=ax)
plot_linear_fit(chinesecal_21425, chinesecal_21425[0:5], 'second_derivative', 'Concentration (mg/L)', .25, 1, .1, ax=ax)

- why don't old and new calibration curves match up when corrected? the correction factor for the old standards seems to be 1.1, not 1.168 as the last standard run would indicate... 

# Ocean S::CAN

In [ ]:
oceancal_12225 = scan_helpers.import_scan_fp('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/SCAN Calibration 1-22-25/calibration.fp')
oceancal_12225['Concentration (mg/L)'] = [0, .01, .05, .1, .5, 1, 2]
oceancal_12225['Concentration (mg/L) Correct'] = oceancal_12225['Concentration (mg/L)']*1.168
oceancal_12225['Concentration (uM)'] = oceancal_12225['Concentration (mg/L) Correct']*1000/14
oceancal_12225 = scan_helpers.apply_calibrations(oceancal_12225, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})

oceancal_12225_corrected = scan_helpers.correct_turbidity(oceancal_12225.drop(['Concentration (mg/L)','Concentration (mg/L) Correct', 'Concentration (uM)', 'one_wavelength', 'two_wavelength', 'second_derivative'], axis=1))
oceancal_12225_corrected['Concentration (mg/L)'] = oceancal_12225['Concentration (mg/L)']
oceancal_12225_corrected['Concentration (mg/L) Correct'] = oceancal_12225['Concentration (mg/L) Correct']
oceancal_12225_corrected['Concentration (uM)'] = oceancal_12225_corrected['Concentration (mg/L) Correct']*1000/14
oceancal_12225_corrected = scan_helpers.apply_calibrations(oceancal_12225_corrected, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})

oceancal_12225_corrected

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(8.5,11))
oceancal_12225.plot(y='Concentration (uM)', x='second_derivative', marker='.', linestyle = 'None', title = 'Uncorrected Second Derivative', ylabel = 'Concentration (uM)', legend=False, ax=ax[0,0])
plot_linear_fit(oceancal_12225, oceancal_12225[0:5], 'second_derivative', 'Concentration (uM)', .25, 60, 10, ax=ax[0,0])

oceancal_12225.plot(y='Concentration (uM)', x='one_wavelength', marker='.', linestyle = 'None', title = 'Uncorrected One Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[1,0])
plot_linear_fit(oceancal_12225, oceancal_12225[0:5], 'one_wavelength', 'Concentration (uM)', .25, 60, 10, ax=ax[1,0])

oceancal_12225.plot(y='Concentration (uM)', x='two_wavelength', marker='.', linestyle = 'None', title = 'Uncorrected Two Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[2,0])
plot_linear_fit(oceancal_12225, oceancal_12225[0:5], 'two_wavelength', 'Concentration (uM)', .25, 60, 10, ax=ax[2,0])

oceancal_12225_corrected.plot(y='Concentration (uM)', x='second_derivative', marker='.', linestyle = 'None', title = 'Corrected Second Derivative', ylabel = 'Concentration (uM)', legend=False, ax=ax[0,1])
plot_linear_fit(oceancal_12225_corrected, oceancal_12225_corrected[0:5], 'second_derivative', 'Concentration (uM)', .25, 60, 10, ax=ax[0,1])

oceancal_12225_corrected.plot(y='Concentration (uM)', x='one_wavelength', marker='.', linestyle = 'None', title = 'Corrected One Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[1,1])
plot_linear_fit(oceancal_12225_corrected, oceancal_12225_corrected[0:5], 'one_wavelength', 'Concentration (uM)', .25, 140, 10, ax=ax[1,1])

oceancal_12225_corrected.plot(y='Concentration (uM)', x='two_wavelength', marker='.', linestyle = 'None', title = 'Corrected Two Wavelength', ylabel = 'Concentration (uM)', legend=False, ax=ax[2,1])
plot_linear_fit(oceancal_12225_corrected, oceancal_12225_corrected[0:5], 'two_wavelength', 'Concentration (uM)', .25, 60, 10, ax=ax[2,1])


fig.suptitle('Ocean S::CAN Calbrations')
fig.tight_layout()

Uncorrected
- calibration for the ocean s::Can (current at RME) also looks good - linear to .5 mg/L, LOD somewhere betweenm .01 and .05 mg/L (expected is .14)
- pretty similar slopes to chinese s::can
- 
Corrected
- similarly small effect for second derivative
- one wavelength corrected is quite bad, 2nd derivative works well